# Synthetic Data Simulation Guide: `blobs` and `normal` Datasets

This guide documents the data simulation pipeline used to generate the two two-dimensional synthetic datasets — **`blobs`** and **`normal`** — that serve as inputs for the KD-tree performance studies. Both datasets are stored in S3 under `s3://jcgs/data/{distribution}/2^{size}/data.parquet` and share the same size ladder and parallelisation strategy; they differ only in how each partition's rows are drawn.

**Environment:** PySpark kernel on AWS EMR (YARN). `spark` and `sc` (`SparkContext`) are available as session globals.

---

## Dataset inventory

Both distributions are generated at the following sizes:

| Size label | Row count | Generation loop |
|---|---|---|
| `2^26` | 67 108 864 | Loop 3 (single cell) |
| `2^27` | 134 217 728 | Loop 2 |
| `2^28` | 268 435 456 | Loop 2 |
| `2^29` | 536 870 912 | Loop 2 |
| `2^30` | 1 073 741 824 | Loop 1 |
| `2^32` | 4 294 967 296 | Loop 1 |
| `2^34` | 17 179 869 184 | Loop 1 |
| `2^36` | 68 719 476 736 | Loop 1 |

---

## Parallelisation strategy (common to both distributions)

Row generation is distributed across Spark partitions. Each partition is assigned an integer index `r` (its seed), and independently generates exactly $2^{20} = 1\,048\,576$ rows by calling `generate_data(r, n=2^{20})`. The total row count for a dataset of size $2^s$ is therefore:

$$2^s = \underbrace{2^p}_{\text{number of partitions}} \times \underbrace{2^{20}}_{\text{rows per partition}}$$

so the seed RDD is `sc.parallelize(arange(2**p))` where $p = s - 20$.

The final RDD is always repartitioned to **5 000** partitions before writing, so the on-disk Parquet layout is uniform regardless of how many generation partitions were used.

For the four largest sizes (Loop 1, $s \in \{30, 32, 34, 36\}$), the seed RDD is additionally repartitioned to $10 \times 192 = 1920$ partitions before `flatMap` to control task granularity. The three medium sizes (Loop 2, $s \in \{27, 28, 29\}$) use no intermediate repartition. The smallest size (Loop 3, $s = 26$) repartitions the seed RDD to **16** partitions before `flatMap`.

The schema for both distributions is:

```
StructType([
    StructField("x", DoubleType()),
    StructField("y", DoubleType())
])
```

Output paths follow the pattern `s3://jcgs/data/{distribution}/2^{size}/data.parquet`.

---


## 1. Imports

Both notebooks share the same two imports. `arange` is used to build the integer seed RDD. `StructType`, `StructField`, and `DoubleType` define the output schema.


In [ ]:
from numpy import arange
from pyspark.sql.types import StructType, StructField, DoubleType

## 2. Row generation functions

### 2a. `generate_data` — `blobs` distribution

Each call generates `n` two-dimensional points drawn from a Gaussian mixture model. The cluster structure is fixed across all partitions and dataset sizes by seeding its RNG at **42**; only the point-level draws vary by partition seed.

**Cluster parameters** (seeded at 42, computed once per call):

| Parameter | Distribution | Shape |
|---|---|---|
| `centers` | $\text{Uniform}(5, 95)$ | $(1000, 2)$ |
| `cluster_std` | $\text{Uniform}(0.5, 2.5)$ | $(1000,)$ |
| `cluster_probs` | Normalised $\text{Exponential}(1)$ weights | $(1000,)$ |

**Point-level generation** (seeded per partition via `seed`):

- $n_{\text{noise}} = \max(1,\ \lfloor n \times 0.02 \rfloor)$ points are drawn uniformly over $[0, 100]^2$.
- $n_{\text{blobs}} = n - n_{\text{noise}}$ points are drawn from the mixture: each point's cluster is sampled from `cluster_probs`, then its coordinates are drawn from $\mathcal{N}(\text{center}, \text{std}^2 \cdot I)$ and clipped to $[0, 100]$.
- Both $x$ and $y$ coordinates use the **same** per-cluster standard deviation (`cluster_std[cid]`), i.e. the blobs are isotropic.
- The two streams (blobs then noise) are concatenated in that order.

Returns a list of `n` `Row("x", "y")` objects.


In [ ]:
def generate_data(seed, n,
                  n_clusters=1000,
                  noise_fraction=0.02,
                  domain_min=0.0, domain_max=100.0):
    from numpy import clip
    from numpy.random import default_rng
    from pyspark.sql import Row

    # Cluster parameters: fixed across all partitions and dataset sizes
    rng_k = default_rng(42)
    centers       = rng_k.uniform(5.0, 95.0, size=(n_clusters, 2))
    cluster_std   = rng_k.uniform(0.5, 2.5,  size=n_clusters)
    raw_w         = rng_k.exponential(1.0,    size=n_clusters)
    cluster_probs = raw_w / raw_w.sum()

    # Point-level generation: one independent stream per partition
    rng     = default_rng(seed)
    n_noise = max(1, int(n * noise_fraction))
    n_blobs = n - n_noise

    cids = rng.choice(n_clusters, size=n_blobs, p=cluster_probs)
    cx, cy = centers[cids, 0], centers[cids, 1]
    sx     = cluster_std[cids]
    bx = clip(rng.normal(cx, sx), domain_min, domain_max)
    by = clip(rng.normal(cy, sx), domain_min, domain_max)

    nx = rng.uniform(domain_min, domain_max, size=n_noise)
    ny = rng.uniform(domain_min, domain_max, size=n_noise)

    xs = list(bx) + list(nx)
    ys = list(by) + list(ny)

    row_type = Row("x", "y")
    rows = [row_type(float(xs[i]), float(ys[i])) for i in range(n)]
    
    return rows

### 2b. `generate_data` — `normal` distribution

Each call generates `n` two-dimensional points drawn from a single bivariate normal distribution. The distribution parameters are fixed (not data-driven) and identical across all partitions and sizes.

**Distribution parameters:**

| Parameter | Value |
|---|---|
| Mean $\mu$ | $(0,\ 0)$ |
| Covariance $\Sigma$ | $\begin{pmatrix} 1 & 0.5 \\ 0.5 & 1 \end{pmatrix}$ |

The legacy `numpy.random.seed` / `numpy.random.multivariate_normal` API is used (rather than `default_rng`). The partition seed is set via `numpy.random.seed(seed)` before drawing.

Returns a list of `n` `Row("x", "y")` objects.


In [ ]:
def generate_data(seed, n):
    from numpy import array, random
    from pyspark.sql import Row

    random.seed(seed)
    mu    = array([0.0, 0.0])
    sigma = array([[1.0, 0.5], [0.5, 1.0]])
    
    x, y = random.multivariate_normal(mean=mu, cov=sigma, size=n).T
    row_type = Row("x", "y")
    rows = [row_type(float(x[i]), float(y[i])) for i in range(n)]
    
    return rows

## 3. Data generation loops

### Loop 1 — four largest sizes: $2^{30},\ 2^{32},\ 2^{34},\ 2^{36}$

Iterates `k` over `range(4)`. For each `k`:

- **Seed RDD:** `arange(2**(10+2*k))` integers, i.e. $2^{10}, 2^{12}, 2^{14}, 2^{16}$ seeds for $k = 0, 1, 2, 3$ respectively.
- **Intermediate repartition:** the seed RDD is repartitioned to $10 \times 192 = 1920$ partitions before `flatMap`.
- **Row count:** $2^{10+2k} \times 2^{20} = 2^{30+2k}$ rows.
- **Output path:** `s3://jcgs/data/{distribution}/2^{30+2*k}/data.parquet`
- Final repartition to **5 000** partitions before writing.

Replace `generate_data` with the blobs or normal version defined above, and `{distribution}` with `blobs` or `normal` accordingly.


In [ ]:
for k in range(4):
    row_rdd = sc.parallelize(arange(2**(10+2*k)).tolist()).repartition(10*192).flatMap(lambda r: generate_data(r, 2**20)).repartition(5000)
    schema = StructType([StructField("x", DoubleType()), StructField("y", DoubleType())])
    data = spark.createDataFrame(row_rdd, schema = schema)
    output_path = f"s3://jcgs/data/{distribution}/2^{30+2*k}/data.parquet"
    data.write.parquet(output_path)

### Loop 2 — three medium sizes: $2^{27},\ 2^{28},\ 2^{29}$

Iterates `k` over `range(3)`. For each `k`:

- **Seed RDD:** `arange(2**(7+k))` integers, i.e. $2^7, 2^8, 2^9$ seeds for $k = 0, 1, 2$ respectively.
- **No intermediate repartition** of the seed RDD before `flatMap`.
- **Row count:** $2^{7+k} \times 2^{20} = 2^{27+k}$ rows.
- **Output path:** `s3://jcgs/data/{distribution}/2^{27+k}/data.parquet`
- Final repartition to **5 000** partitions before writing.


In [ ]:
for k in range(3):
    row_rdd = sc.parallelize(arange(2**(7+k)).tolist()).flatMap(lambda r: generate_data(r, 2**20)).repartition(5000)
    schema = StructType([StructField("x", DoubleType()), StructField("y", DoubleType())])
    data = spark.createDataFrame(row_rdd, schema = schema)
    output_path = f"s3://jcgs/data/{distribution}/2^{27+k}/data.parquet"
    data.write.parquet(output_path)

### Loop 3 — smallest size: $2^{26}$

A single standalone cell (no loop):

- **Seed RDD:** `arange(2**6)` = 64 seeds.
- **Intermediate repartition:** the seed RDD is repartitioned to **16** partitions before `flatMap`.
- **Row count:** $2^6 \times 2^{20} = 2^{26}$ rows.
- **Output path:** `s3://jcgs/data/{distribution}/2^26/data.parquet`
- Final repartition to **5 000** partitions before writing.


In [ ]:
row_rdd = sc.parallelize(arange(2**6).tolist()).repartition(16).flatMap(lambda r: generate_data(r, 2**20)).repartition(5000)
schema = StructType([StructField("x", DoubleType()), StructField("y", DoubleType())])
data = spark.createDataFrame(row_rdd, schema = schema)
output_path = f"s3://jcgs/data/{distribution}/2^26/data.parquet"
data.write.parquet(output_path)